<a href="https://colab.research.google.com/github/mukulhayaran/pyTorch-Learning/blob/main/pyTorch_dataloaders_for_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


 import torch
 import torch.nn
 import torchvision
 from torchvision import transforms
 from torch.utils.data import Dataset,DataLoader
 from torchvision.datasets import ImageFolder #stream data from images stored in folders

 import os #allows access to files
 import numpy as np
 from PIL import Image # helps load images
 from collections import Counter #gives count of unique items in an iterable

Build Tokenizer Dictionary

In [ ]:
# Download the raw dataset archive from Stanford's servers
!wget http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz

# Extract the archive (this might take a minute as there are 50,000 text files)
!tar -xzf aclImdb_v1.tar.gz

In [ ]:
from tqdm.notebook import tqdm



path_to_data="aclImdb/train"
os.listdir(path_to_data)
path_to_pos_field=os.path.join(path_to_data,"pos")
path_to_neg_field=os.path.join(path_to_data,"neg")

path_to_pos_txt=[os.path.join(path_to_pos_field,file) for file in os.listdir(path_to_pos_field)]
path_to_neg_txt=[os.path.join(path_to_neg_field,file) for file in os.listdir(path_to_neg_field)]

training_files= path_to_pos_txt + path_to_neg_txt

alltxt=""
for file in tqdm(training_files):
  with open(file,"r") as f:
    text= f.readlines()[0]
    alltxt+=text






In [ ]:
unique_counts=dict(Counter(alltxt))
characters=sorted([key for (key,value) in unique_counts.items() if value> 1500])
characters
characters.append("<UNK>")
characters.append("<PAD>")


char2idx={}
idx2char={}

for idx,char in enumerate(characters):
  char2idx[char]=idx
  idx2char[idx]=char




In [ ]:
class IMDBDataset(Dataset):

  def __init__(self, path_to_data,char2idx):
    path_to_pos_field=os.path.join(path_to_data,"pos")
    path_to_neg_field=os.path.join(path_to_data,"neg")

    path_to_pos_txt=[os.path.join(path_to_pos_field,file) for file in os.listdir(path_to_pos_field)]
    path_to_neg_txt=[os.path.join(path_to_neg_field,file) for file in os.listdir(path_to_neg_field)]

    self.training_files= path_to_pos_txt + path_to_neg_txt

    self.tokenizer=char2idx
    self.pos_label=1
    self.neg_label=0



  def __len__(self):
    return len(self.training_files)

  def __getitem__(self,idx):

    path_to_txt = self.training_files[idx]
    with open(path_to_txt,"r") as f:
      text= f.readlines()[0]

    tokenized=[]

    for char in text:
      if char in self.tokenizer.keys():
        tokenized.append(self.tokenizer[char])

      else:
        tokenized.append(self.tokenizer["<UNK>"])

    sample=torch.tensor(tokeized)

    label= self.pos_label if "pos" in path_to_txt else self.neg_label
    return sample,label

path_to_data="aclImdb/train"
dataset=IMDBDataset(path_to_data,char2idx)
#loader= DataLoader(dataset, batch_size=16) will give error because samples are of different length




Data Collator:


In [ ]:
#data collator: what the data loader uses to stick the samples together

def data_collator(batch):
  texts, labels=[] , []

  for text,label in batch:
    texts.append(text)
    labels.append(label)

  labels=torch.tensor(labels)
  texts=torch.nn.utils.rnn.pad_sequence(texts, batch_first=True)

  return texts,labels

In [ ]:
loader= DataLoader(dataset, batch_size=4, collate_fn=data_collator)

for texts, labels in loader:
  print(texts)
  break